# Pipeline

https://github.com/datamindedbe/blog-tpcds-dbt-duckdb/tree/main

```
uv sync
```

In [ ]:
# # run this to generate index for values in the hierarchy yaml files

# import duckdb
# from src.hierarchy_duckdb import build_tree_with_stats
# from pathlib import Path
# proj_path = Path().resolve()
# data_path = proj_path / 'data'
# duckdb_conn = duckdb.connect(database=str(proj_path / 'tpcds/tpcds.db'))
# index_path = data_path / 'index'
# for yaml_path in (data_path / 'hierarchy').glob('*.yaml'):
#     tree = build_tree_with_stats(yaml_path, index_path, duckdb_conn)
#     with (data_path / 'hierarchy' / f"{yaml_path.stem}.json").open('w') as f:
#         f.write(tree.to_json())

In [ ]:
# # run this only once to generate the TPC-DS data
import duckdb

con = duckdb.connect(database='./tpcds/tpcds.db')
# con = duckdb.connect(
#     database='./cube-project/data/tpcds.db',
#     read_only=True,
# )
# con.execute('INSTALL tpcds;')
# con.execute('LOAD tpcds;')
# con.execute("CALL dsdgen(sf = 1);")  # run only once generate data with scale factor 1 (1GB)

In [3]:
df = con.execute("""SELECT ca_zip, Sum(cs_sales_price) 
FROM   catalog_sales, 
       customer, 
       customer_address, 
       date_dim 
WHERE  cs_bill_customer_sk = c_customer_sk 
       AND c_current_addr_sk = ca_address_sk 
       AND ( Substr(ca_zip, 1, 5) IN ( '85669', '86197', '88274', '83405', 
                                       '86475', '85392', '85460', '80348', 
                                       '81792' ) 
              OR ca_state IN ( 'CA', 'WA', 'GA' ) 
              OR cs_sales_price > 500 ) 
       AND cs_sold_date_sk = d_date_sk 
       AND d_qoy = 1 
       AND d_year = 1998 
GROUP  BY ca_zip 
ORDER  BY ca_zip
LIMIT 100; 
""").fetch_df()
df.head()

,ca_zip,sum(cs_sales_price)
0,30069,1603.94
1,30150,1030.54
2,30162,408.90
3,30169,855.10
4,30399,224.72


Ghita

In [4]:
df = con.execute("""
WITH customer_total_return 
     AS (SELECT sr_customer_sk     AS ctr_customer_sk, 
                sr_store_sk        AS ctr_store_sk, 
                Sum(sr_return_amt) AS ctr_total_return 
         FROM   store_returns, 
                date_dim 
         WHERE  sr_returned_date_sk = d_date_sk 
                AND d_year = 2001 
         GROUP  BY sr_customer_sk, 
                   sr_store_sk) 
SELECT c_customer_id 
FROM   customer_total_return ctr1, 
       store, 
       customer 
WHERE  ctr1.ctr_total_return > (SELECT Avg(ctr_total_return) * 1.2 
                                FROM   customer_total_return ctr2 
                                WHERE  ctr1.ctr_store_sk = ctr2.ctr_store_sk) 
       AND s_store_sk = ctr1.ctr_store_sk 
       AND s_state = 'TN' 
       AND ctr1.ctr_customer_sk = c_customer_sk 
ORDER  BY c_customer_id
LIMIT 100;
""").fetch_df()
df.head()

,c_customer_id
0,AAAAAAAAAAAOAAAA
1,AAAAAAAAAABDAAAA
2,AAAAAAAAAABDBAAA
3,AAAAAAAAAACEAAAA
4,AAAAAAAAAACPAAAA


# Schema Graph

In [1]:
import sys
from pathlib import Path
proj_path = Path().resolve()
sys.path.append(str(proj_path / 'src'))

from src.schema_processor import display_graph
data_path = proj_path / 'data'

In [2]:
display_graph(data_path)

Output()

In [ ]:
from src.schema_processor import SchemaExplorer

explorer = SchemaExplorer(data_path)
print(explorer.get_facts())
print(explorer.get_schema('star', 'catalog_sales'))
print()
# TODO: need from/target searching
attr_results = explorer.search_attribute('star', 'catalog_sales', 'ca_state')  # d_fy_year, s_store_id
attr_results

In [ ]:
import random
from src.unique_index import UniqueIndex
path = './data/index/date_dim__d_fy_year'
idx = UniqueIndex(path, fast=False)

x = random.sample(list(iter(idx)), k=1)[0]
print("Search for:", x)
o = explorer.search_value('star', 'store_sales', 'd_fy_year', x)
print("Found:", o[0]['found'])
o

----

## About the TPC-DS queries

In [ ]:
from pathlib import Path
from collections import defaultdict
from src.schema_processor import SchemaExplorer
queries_path = Path('./queries/tpcds')

fact2queries = defaultdict(list)
queries2fact = defaultdict(set)
for qpath in queries_path.glob('*.sql'):
    with qpath.open() as f:
        sql = f.read()
    for fact in SchemaExplorer.tpcds_facts:
        if fact in sql.lower():
            queries2fact[qpath.stem].add(fact)

for query, facts in queries2fact.items():
    # set the number of the facts as key, more or equal to 4 make them one group
    if len(facts) >= 4:
        fact2queries['4+'].append(query)
    else:
        fact2queries[str(len(facts))].append(query)

In [ ]:
for k, v in sorted(fact2queries.items(), key=lambda x: (int(x[0]) if x[0].isdigit() else 99)):
    print(f"{k}: {len(v)}")

In [ ]:
queries2fact['query15']

In [ ]:
facts_queries_by_numbers: dict[str, dict[str, list[str]]] = defaultdict(dict)
for fact in SchemaExplorer.tpcds_facts:
    for number, queries in fact2queries.items():
        if facts_queries_by_numbers[fact].get(number) is None:
            facts_queries_by_numbers[fact][number] = []
        facts_queries_by_numbers[fact][number].extend(queries)

In [ ]:
sorted(facts_queries_by_numbers['store_sales']['1'])[:5]

In [ ]:
# plot the distribution of number of fact tables per query
# make the number in the center of the bars
# if number is 4 or more, put it in 4+
import matplotlib.pyplot as plt
import seaborn as sns
plt.style.use('ggplot')

fact_counts = [len(v) for v in queries2fact.values()]
fact_counts = [4 if x >= 4 else x for x in fact_counts]
fig, ax = plt.subplots()

sns.histplot(fact_counts, discrete=True, shrink=0.9, ax=ax)
ax.set_xlabel('Number of Fact Tables in a Query')
ax.set_ylabel('Number of Queries')
ax.set_title('Distribution of Number of Fact Tables per Query')
plt.xticks(ticks=[1, 2, 3, 4], labels=['1', '2', '3', '4+'])
plt.grid(axis='y')
plt.show()

UI API for **Query 15**

```json
{
  "measures": ["catalog_sales.cs_sales_price"],
  "dimensions": ["customer_address.ca_zip"],
  "filters": [
    {
      "or": [
        {
          "member": "customer_address.ca_zip",
          "operator": "startsWith",
          "values": ["85669","86197","88274","83405","86475","85392","85460","80348","81792"]
        },
        {
          "member": "customer_address.ca_state",
          "operator": "equals",
          "values": ["CA","WA","GA"]
        },
        {
          "member": "catalog_sales.cs_sales_price_raw",
          "operator": "gt",
          "values": ["500"]
        }
      ]
    },
    { "member": "date_dim_sold.d_year", "operator": "equals", "values": ["1998"] },
    { "member": "date_dim_sold.d_qoy",  "operator": "equals", "values": ["1"] }
  ],
  "order": { "customer_address_billing.ca_zip": "asc" },
  "limit": 100
}
```

```
We need to convert the sqls in the queries/tpcds into REST API of the Cube.dev. Please convert the SQL into queries from query1.sql to query10.sql. The example is queries/tpcds/query15.sql to queries/rest_api/query15.json.
If you don't know the query format, the documentation is in https://cube.dev/docs/product/apis-integrations/rest-api/query-format. Let me attach the summary for you.
```Query Properties
A Query has the following properties:

measures: An array of measures.
dimensions: An array of dimensions.
filters: An array of objects, describing filters. Learn about filters format.
timeDimensions: A convenient way to specify a time dimension with a filter. It is an array of objects in timeDimension format.
segments: An array of segments. A segment is a named filter, created in the data model.
limit: A row limit for your query.
total: If set to true, Cube will run a total query and return the total number of rows as if no row limit or offset are set in the query. The default value is false.
provided, default ordering is applied. If an empty object ([]) is provided, no ordering is applied.
timezone: A time zone for your query. You can set the desired time zone in the TZ Database Name(opens in a new tab) format, e.g., America/Los_Angeles.
renewQuery: If renewQuery is set to true, Cube will renew all refreshKey for queries and query results in the foreground. However, if the refreshKey (or refreshKey.every) doesn't indicate that there's a need for an update
this setting has no effect. The default value is false.
NOTE: Cube provides only eventual consistency guarantee. Using a small refreshKey.every value together with renewQuery to achieve immediate consistency can lead to endless refresh loops and overall system instability.
ungrouped: If set to true, Cube will run an ungrouped query.
joinHints: Query-time join hints, provided as an array of two-element arrays of cube names.```.

If you need the schema then check the model definitions in the cube-project/cube_conf/model/cubes.
If it is tricky, you need to add the proper schema(columns) in the measures, dimensions, filters, timeDimensions, segments etc in the yaml files. 
Only add do not remove anything from the yaml files.
The table schema is in the data/tables.
```